<a href="https://colab.research.google.com/github/HooroBoy/Pemes/blob/main/JS06/JS06_TugasLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. Load Data
df_voice = pd.read_csv('voice.csv')

# Pisahkan fitur dan label
X = df_voice.iloc[:, :-1]
y = df_voice.iloc[:, -1]

# Encode label (misal 'male'/'female' menjadi 1/0) jika masih bertipe string
if y.dtype == 'object':
    le = LabelEncoder()
    y = le.fit_transform(y)

# Standarisasi fitur (sangat krusial untuk algoritma SVM)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Pengujian dengan variasi split dan kernel
splits = [
    {'test_size': 0.30, 'label': '70:30'},
    {'test_size': 0.20, 'label': '80:20'}
]

kernels = ['linear', 'poly', 'rbf']
results = []

for s in splits:
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=s['test_size'], random_state=42, stratify=y
    )

    for k in kernels:
        # Inisialisasi dan training model SVM
        model = SVC(kernel=k, random_state=42)
        model.fit(X_train, y_train)

        # Evaluasi
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        results.append({
            'Rasio Split': s['label'],
            'Kernel': k,
            'Akurasi (%)': round(acc * 100, 2)
        })

# 3. Tabulasi Performa
df_results = pd.DataFrame(results)
print("=== Performa SVM pada voice.csv ===")
display(df_results)

=== Performa SVM pada voice.csv ===


,Rasio Split,Kernel,Akurasi (%)
0,70:30,linear,97.90
1,70:30,poly,95.90
2,70:30,rbf,98.32
3,80:20,linear,97.48
4,80:20,poly,95.74
5,80:20,rbf,98.26


In [4]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.image as mpimg
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Fungsi Load Dataset Gambar
def load_all_images(base_dirs):
    img_list = []
    for b_dir in base_dirs:
        p = Path(b_dir)
        for class_dir in p.glob('*'):
            if class_dir.is_dir():
                label = class_dir.name  # 'day' atau 'night'
                for file in class_dir.glob('*.jpg'):
                    img = mpimg.imread(str(file))
                    if img is not None:
                        img_list.append((img, label))
    return img_list

# Path direktori sesuai praktikum 5
train_dir = "/content/drive/MyDrive/Pemes/images/training"
test_dir = "/content/drive/MyDrive/Pemes/images/test"

raw_dataset = load_all_images([train_dir, test_dir])
print(f"Total gambar yang dimuat: {len(raw_dataset)}")

# 2. Ekstraksi Fitur Histogram Warna (HSV)
def extract_color_histogram(image, bins=(8, 8, 8)):
    # Resize untuk konsistensi dimensi
    img_resized = cv2.resize(image, (256, 256))
    # Konversi ruang warna ke HSV
    img_hsv = cv2.cvtColor(img_resized, cv2.COLOR_RGB2HSV)

    # Hitung 3D histogram untuk channel H, S, V
    hist = cv2.calcHist([img_hsv], [0, 1, 2], None, bins, [0, 180, 0, 256, 0, 256])
    # Normalisasi histogram agar invariant terhadap variasi ukuran/pencahayaan absolut
    cv2.normalize(hist, hist)

    return hist.flatten()

X_features = []
y_labels = []

for img, label in raw_dataset:
    hist_features = extract_color_histogram(img)
    X_features.append(hist_features)
    y_labels.append(1 if label == 'day' else 0)

X_features = np.array(X_features)
y_labels = np.array(y_labels)

# 3. Train-Test Split (80:20)
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_features, y_labels, test_size=0.20, random_state=42, stratify=y_labels
)

# Standarisasi fitur histogram
scaler_img = StandardScaler()
X_train_scaled = scaler_img.fit_transform(X_train_img)
X_test_scaled = scaler_img.transform(X_test_img)

# 4. Hyperparameter Tuning Kernel RBF dengan GridSearchCV
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
}

grid_search = GridSearchCV(
    estimator=SVC(kernel='rbf', random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train_img)

# 5. Evaluasi Model Terbaik
best_model = grid_search.best_estimator_
y_pred_img = best_model.predict(X_test_scaled)
test_acc = accuracy_score(y_test_img, y_pred_img)

# Tampilkan Hasil
print("=== Hasil Tuning & Evaluasi Model SVM RBF Citra Siang/Malam ===")
print(f"Hyperparameter Terbaik : {grid_search.best_params_}")
print(f"Akurasi Validasi (CV)  : {grid_search.best_score_ * 100:.2f}%")
print(f"Akurasi Data Uji (20%) : {test_acc * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test_img, y_pred_img, target_names=['night (0)', 'day (1)']))

# Ringkasan Performa dalam Bentuk Tabel
df_summary = pd.DataFrame([{
    'Dataset': 'Siang vs Malam',
    'Metode Ekstraksi': 'Color Histogram (HSV 3D)',
    'Split': '80:20',
    'Kernel': 'RBF',
    'Best C': grid_search.best_params_['C'],
    'Best Gamma': grid_search.best_params_['gamma'],
    'Akurasi Uji (%)': round(test_acc * 100, 2)
}])
display(df_summary)

Total gambar yang dimuat: 400
=== Hasil Tuning & Evaluasi Model SVM RBF Citra Siang/Malam ===
Hyperparameter Terbaik : {'C': 10, 'gamma': 0.001}
Akurasi Validasi (CV)  : 98.12%
Akurasi Data Uji (20%) : 97.50%

Classification Report:
              precision    recall  f1-score   support

   night (0)       0.95      1.00      0.98        40
     day (1)       1.00      0.95      0.97        40

    accuracy                           0.97        80
   macro avg       0.98      0.97      0.97        80
weighted avg       0.98      0.97      0.97        80



,Dataset,Metode Ekstraksi,Split,Kernel,Best C,Best Gamma,Akurasi Uji (%)
0,Siang vs Malam,Color Histogram (HSV 3D),80:20,RBF,10,0.001,97.5
